# soramado — 多重散乱LUT事前計算 / Multiple-Scattering LUT Precomputation

`web/` アプリ用の **多重散乱を含む大気散乱LUT** をGPU(なければCPU)で事前計算し、
`manifest.json` + `transmittance.bin` + `scattering.bin`(float32, little-endian)+ プレビューPNG として書き出します。

Precomputes atmospheric-scattering LUTs **including multiple scattering** and exports them
for the web app. Single scattering is integrated exactly; higher orders use the
isotropic multiple-scattering approximation of Hillaire 2020 (EGSR), built on the
Bruneton & Neyret 2008 single-scattering formulation.

## 使い方 / How to run
1. ランタイム → ランタイムのタイプを変更 → **GPU**(T4で十分。CPUでも数分で完走)
2. すべてのセルを実行(Runtime → Run all)
3. 最後に `soramado_lut.zip` がダウンロードされるので、展開して中身を **`web/public/lut/`** に配置
4. アプリは起動時に `/lut/manifest.json` を見つけると自動で多重散乱LUTモードに切り替わります(無ければ単一散乱のリアルタイム計算にフォールバック)

## 出力フォーマット / Output format
- `transmittance.bin` — float32 RGB, shape **(height=64, width=256, 3)**(x = mu が最内)
- `scattering.bin` — float32 RGB, shape **(nu=32, muS=64, mu=128, 3)**(x = mu が最内)
  - 値は「太陽の大気圏外放射照度 = 1 とした、位相関数込み・全散乱次数合計の放射輝度」
- `manifest.json` — 寸法とスケールの記述

## パラメータ化(シェーダと厳密一致必須)/ Parameterisation (must match `web/src/shaders/sky.frag.glsl`)
テクセル `i`(サイズ `N`)は `u = i / (N - 1)` で評価する(シェーダ側 `lutCoord()` がテクセル中心に補正)。

| axis | mapping (u → physical) |
|---|---|
| scattering `mu`(視線天頂余弦, 地上視点) | `mu = u^3` |
| scattering `muS`(太陽天頂余弦) | `muS = -(ln(1 - u·(1-e^{-3.6})) + 0.6) / 3` ∈ [-0.2, 1] |
| scattering `nu`(視線・太陽の余弦) | `nu = 2u - 1`(物理的に不可能な nu は最近接にクランプして評価) |
| transmittance `mu` | `mu = 1.3u - 0.3` |
| transmittance `h`(高度) | `h = u² · (Rt - Rg)` |

> 将来のニューラル空モデル(Colabで学習 → テクスチャ/ONNX化)も、Webアプリ側の
> `web/src/atmosphere/lut.ts` の `SkySource` インターフェースから同様に差し替え可能です。


In [ ]:
# 環境セットアップ: GPUがあれば cupy、無ければ numpy で動きます
import numpy as np

try:
    import cupy as xp  # Colab GPUランタイムなら高速
    xp.zeros(1)  # GPUが実際に使えるか確認
    GPU = True
    print('backend: cupy (GPU)')
except Exception:
    import numpy as xp
    GPU = False
    print('backend: numpy (CPU) — それでも数分で完走します')

def asnumpy(a):
    return xp.asnumpy(a) if GPU else a


In [ ]:
# ---------------- 物理定数(web/src/shaders/sky.frag.glsl と一致させること) ----------------
Rg = 6360.0e3          # 地表半径 [m]
Rt = 6460.0e3          # 大気上端半径 [m]
Hr = 8500.0            # レイリー散乱スケールハイト [m]
Hm = 1200.0            # ミー散乱スケールハイト [m]

# レイリー散乱係数: βR(λ) = 8π³(n²-1)²/(3Nλ⁴)·(6+3p)/(6-7p) — λ⁻⁴則から導出
def rayleigh_beta(lambdas=(680e-9, 550e-9, 440e-9)):
    n, N, p = 1.000293, 2.547e25, 0.035
    king = (6 + 3*p) / (6 - 7*p)
    f = 8 * np.pi**3 * (n*n - 1)**2 * king / (3 * N)
    return np.array([f / l**4 for l in lambdas])

BETA_R  = xp.asarray(rayleigh_beta())            # 散乱 = 消散(レイリー)
BETA_MS = xp.asarray([3.996e-6] * 3)             # ミー散乱
BETA_ME = xp.asarray([4.440e-6] * 3)             # ミー消散
MIE_G   = 0.8
BETA_O3 = xp.asarray([0.650e-6, 1.881e-6, 0.085e-6])  # オゾン吸収(薄明の藍に必須)
O3_CENTER, O3_HALFW = 25.0e3, 15.0e3
GROUND_ALBEDO = 0.1                              # 地表反射(多重散乱の地面バウンス)

# ---------------- LUT寸法 ----------------
T_W, T_H = 256, 64        # transmittance: (mu, altitude)
S_MU, S_MUS, S_NU = 128, 64, 32   # scattering: 3Dテクスチャ
N_VIEW_STEPS  = 64        # 最終ビュー積分のステップ数
MS_NH, MS_NMUS = 32, 32   # Ψms(多重散乱)2Dグリッド
MS_NDIR, MS_NSTEP = 64, 24

print('beta_R =', asnumpy(BETA_R))


In [ ]:
# ---------------- 基本関数 ----------------
def densities(h):
    """(rayleigh, mie, ozone) の相対密度。h: 高度[m] 配列"""
    h = xp.maximum(h, 0.0)
    dr = xp.exp(-h / Hr)
    dm = xp.exp(-h / Hm)
    do = xp.maximum(0.0, 1.0 - xp.abs(h - O3_CENTER) / O3_HALFW)
    return dr, dm, do

def dist_to_top(r, mu):
    """半径r・天頂余弦muの視線が大気上端 Rt に達するまでの距離"""
    disc = xp.maximum(r*r*(mu*mu - 1.0) + Rt*Rt, 0.0)
    return xp.maximum(-r*mu + xp.sqrt(disc), 0.0)

def hits_ground(r, mu):
    disc = r*r*(mu*mu - 1.0) + Rg*Rg
    return (mu < 0) & (disc >= 0)

def transmittance_integrate(r, mu, n_steps=256):
    """T(r,mu): 大気上端までの透過率(RGB)。地面に当たる視線は0。任意shapeの r, mu に対応"""
    L = dist_to_top(r, mu)
    t = (xp.arange(n_steps) + 0.5) / n_steps          # (S,)
    seg = L[..., None] * t                            # (..., S)
    dl  = (L / n_steps)[..., None]
    rs  = xp.sqrt(r[..., None]**2 + seg**2 + 2.0*r[..., None]*seg*mu[..., None])
    dr, dm, do = densities(rs - Rg)
    od_r = (dr * dl).sum(-1); od_m = (dm * dl).sum(-1); od_o = (do * dl).sum(-1)
    tau = (BETA_R[None] * od_r[..., None] + BETA_ME[None] * od_m[..., None]
           + BETA_O3[None] * od_o[..., None])
    T = xp.exp(-tau)
    T[hits_ground(r, mu)] = 0.0
    return T   # (..., 3)

def phase_rayleigh(c):
    return 3.0 / (16.0 * np.pi) * (1.0 + c*c)

def phase_mie(c, g=MIE_G):
    return (1.0 - g*g) / (4.0 * np.pi * (1.0 + g*g - 2.0*g*c) ** 1.5)


In [ ]:
# ---------------- 内部用 透過率テーブル(高速参照のためのキャッシュ) ----------------
TT_NR, TT_NMU = 128, 512
_tt_ur = (xp.arange(TT_NR) / (TT_NR - 1.0))
_tt_r  = Rg + _tt_ur**2 * (Rt - Rg)
_tt_mu = -1.0 + 2.0 * (xp.arange(TT_NMU) / (TT_NMU - 1.0))
_R, _MU = xp.meshgrid(_tt_r, _tt_mu, indexing='ij')
TT = transmittance_integrate(_R, _MU)             # (NR, NMU, 3)
print('internal transmittance table:', TT.shape)

def transmittance_lookup(r, mu):
    """バイリニア補間で T(r, mu) を引く(任意shape)"""
    ur = xp.sqrt(xp.clip((r - Rg) / (Rt - Rg), 0.0, 1.0)) * (TT_NR - 1)
    um = (xp.clip(mu, -1.0, 1.0) + 1.0) * 0.5 * (TT_NMU - 1)
    i0 = xp.clip(xp.floor(ur).astype(xp.int32), 0, TT_NR - 2)
    j0 = xp.clip(xp.floor(um).astype(xp.int32), 0, TT_NMU - 2)
    fr = (ur - i0)[..., None]; fm = (um - j0)[..., None]
    t00 = TT[i0, j0]; t10 = TT[i0 + 1, j0]; t01 = TT[i0, j0 + 1]; t11 = TT[i0 + 1, j0 + 1]
    return (t00*(1-fr)*(1-fm) + t10*fr*(1-fm) + t01*(1-fr)*fm + t11*fr*fm)


In [ ]:
# ---------------- Ψms: 等方多重散乱項(Hillaire 2020 §5.2) ----------------
# 各 (高度, 太陽天頂角) について
#   L2 = (1/4π)∮dω ∫dt T(x→p)·[σsR·phR + σsM·phM](sun·ω)·T_sun(p)   (2次散乱)
#   F  = (1/4π)∮dω ∫dt T(x→p)·σs_tot(p)                              (再散乱率)
#   Ψms = L2 / (1 - F)    (無限次数の等比級数・等方近似)
# 地面ヒット時はランバート反射 (albedo/π)·muS·T_sun を加算。

def fibonacci_sphere(n):
    i = xp.arange(n) + 0.5
    phi = np.pi * (3.0 - np.sqrt(5.0)) * i
    z = 1.0 - 2.0 * i / n
    s = xp.sqrt(xp.maximum(0.0, 1.0 - z*z))
    return xp.stack([s * xp.cos(phi), s * xp.sin(phi), z], -1)  # (N,3) z=up

ms_uh   = xp.arange(MS_NH) / (MS_NH - 1.0)
ms_r    = Rg + ms_uh**2 * (Rt - Rg) + 1.0
ms_umus = xp.arange(MS_NMUS) / (MS_NMUS - 1.0)
ms_mus  = xp.clip(-(xp.log(1.0 - ms_umus * (1.0 - np.e**-3.6)) + 0.6) / 3.0, -1.0, 1.0)

DIRS = fibonacci_sphere(MS_NDIR)                  # (D,3)
R4   = ms_r[:, None, None, None]                  # (H,1,1,1)
MUS4 = ms_mus[None, :, None, None]                # (1,S,1,1)
MUD  = DIRS[None, None, :, 2:3]                   # (1,1,D,1) 方向の天頂余弦

gnd  = hits_ground(xp.broadcast_to(R4[..., 0], (MS_NH, MS_NMUS, MS_NDIR)),
                   xp.broadcast_to(MUD[..., 0], (MS_NH, MS_NMUS, MS_NDIR)))
disc_g = xp.maximum(R4[..., 0]**2 * (MUD[..., 0]**2 - 1.0) + Rg*Rg, 0.0)
t_end  = xp.where(gnd, -R4[..., 0]*MUD[..., 0] - xp.sqrt(disc_g),
                  dist_to_top(xp.broadcast_to(R4[..., 0], gnd.shape),
                              xp.broadcast_to(MUD[..., 0], gnd.shape)))  # (H,S,D)

tt   = (xp.arange(MS_NSTEP) + 0.5) / MS_NSTEP
seg  = t_end[..., None] * tt                      # (H,S,D,T)
dl   = (t_end / MS_NSTEP)[..., None]
rp   = xp.sqrt(R4**2 + seg**2 + 2.0*R4*seg*MUD)   # (H,S,D,T) サンプル点の半径
# 太陽天頂余弦@サンプル点: cosθ = (r·muS + t·(s·ω)) / r(t)。s·ω は方向ごとに一定:
# 太陽は天頂角 acos(muS)・方位0に置く → s = (sinθs, 0, muS)
sin_s = xp.sqrt(xp.maximum(0.0, 1.0 - ms_mus**2))[None, :, None, None]
s_dot_w = sin_s * DIRS[None, None, :, 0:1] + MUS4 * DIRS[None, None, :, 2:3]  # (1,S,D,1)
mus_p = (R4 * MUS4 + seg * s_dot_w) / rp

dr, dm, do = densities(rp - Rg)
dlx = dl[..., None]
# 経路途中までの光学的深さ(累積; 中点法)
od_r = xp.cumsum(dr, -1) - 0.5*dr; od_m = xp.cumsum(dm, -1) - 0.5*dm; od_o = xp.cumsum(do, -1) - 0.5*do
tau  = (BETA_R*(od_r*dl)[..., None] + BETA_ME*(od_m*dl)[..., None] + BETA_O3*(od_o*dl)[..., None])
T_path = xp.exp(-tau)                              # (H,S,D,T,3)
T_sun  = transmittance_lookup(rp, mus_p)           # (H,S,D,T,3)

phR = phase_rayleigh(s_dot_w)
phM = phase_mie(s_dot_w)
sigma_s = BETA_R*dr[..., None] + BETA_MS*dm[..., None]                 # σs(p)
src2    = (BETA_R*dr[..., None]*phR[..., None] +
           BETA_MS*dm[..., None]*phM[..., None]) * T_sun               # 2次散乱ソース

L2 = (T_path * src2     * dlx).sum(3).mean(2)      # (H,S,3) 球面平均 = (1/4π)∮
F  = (T_path * sigma_s  * dlx).sum(3).mean(2)

# 地面バウンス(ヒットする方向のみ): T(x→ground)·(albedo/π)·muS(ground)·T_sun(ground)
r_g   = xp.full_like(t_end, Rg + 1.0)
mus_g = (R4[..., 0] * MUS4[..., 0] + t_end * s_dot_w[..., 0]) / r_g
T_to_g  = T_path[..., -1, :]
T_sun_g = transmittance_lookup(r_g, mus_g)
ground  = (T_to_g * T_sun_g * (GROUND_ALBEDO/np.pi) *
           xp.maximum(mus_g, 0.0)[..., None]) * gnd[..., None]
L2 = L2 + ground.mean(2)

PSI_MS = L2 / xp.maximum(1.0 - F, 0.05)            # (MS_NH, MS_NMUS, 3)
print('Psi_ms table:', PSI_MS.shape, ' max =', float(asnumpy(PSI_MS.max())))

def psi_lookup(r, mus):
    uh = xp.sqrt(xp.clip((r - Rg) / (Rt - Rg), 0.0, 1.0)) * (MS_NH - 1)
    us = xp.clip((1.0 - xp.exp(-3.0*xp.clip(mus, -1, 1) - 0.6)) / (1.0 - np.e**-3.6), 0.0, 1.0) * (MS_NMUS - 1)
    i0 = xp.clip(xp.floor(uh).astype(xp.int32), 0, MS_NH - 2)
    j0 = xp.clip(xp.floor(us).astype(xp.int32), 0, MS_NMUS - 2)
    fr = (uh - i0)[..., None]; fm = (us - j0)[..., None]
    p00 = PSI_MS[i0, j0]; p10 = PSI_MS[i0+1, j0]; p01 = PSI_MS[i0, j0+1]; p11 = PSI_MS[i0+1, j0+1]
    return p00*(1-fr)*(1-fm) + p10*fr*(1-fm) + p01*(1-fr)*fm + p11*fr*fm


In [ ]:
# ---------------- 最終LUT: 地上視点の空放射輝度 (mu, muS, nu) ----------------
# L(mu,muS,nu) = ∫ T_view(t)·[ σsR·phR(nu)·T_sun + σsM·phM(nu)·T_sun + σs_tot·Ψms ] dt
# (太陽放射照度=1。シェーダ側で uSunIrradiance を乗算)

u_mu  = xp.arange(S_MU)  / (S_MU  - 1.0)
u_mus = xp.arange(S_MUS) / (S_MUS - 1.0)
u_nu  = xp.arange(S_NU)  / (S_NU  - 1.0)

mu_v  = u_mu ** 3                                          # 視線天頂余弦 [0,1]
mus_v = xp.clip(-(xp.log(1.0 - u_mus*(1.0 - np.e**-3.6)) + 0.6) / 3.0, -1.0, 1.0)
nu_v  = 2.0*u_nu - 1.0

scattering = xp.zeros((S_NU, S_MUS, S_MU, 3), dtype=xp.float64)
r0 = Rg + 2.0   # 地上2m
tt = (xp.arange(N_VIEW_STEPS) + 0.5) / N_VIEW_STEPS

for j, mus in enumerate(asnumpy(mus_v)):
    mus = float(mus)
    MU = mu_v[None, :]                                     # (1,M)
    NU = nu_v[:, None]                                     # (N,1)
    # 物理的に可能な nu の範囲にクランプ(LUT補間の安定化)
    lim = xp.sqrt(xp.maximum(0.0, (1.0 - MU**2))) * np.sqrt(max(0.0, 1.0 - mus**2))
    NUc = xp.clip(NU, mus*MU - lim, mus*MU + lim)          # (N,M)

    L_top = dist_to_top(xp.full_like(MU, r0), MU)          # (1,M)
    seg = L_top[..., None] * tt                            # (1,M,T)→broadcast (N,M,T)
    dl  = (L_top / N_VIEW_STEPS)[..., None]
    rp  = xp.sqrt(r0**2 + seg**2 + 2.0*r0*seg*MU[..., None])
    mus_p = (r0*mus + seg*NUc[..., None]) / rp             # (N,M,T)
    dr, dm, do = densities(rp - Rg)

    od_r = xp.cumsum(dr, -1) - 0.5*dr
    od_m = xp.cumsum(dm, -1) - 0.5*dm
    od_o = xp.cumsum(do, -1) - 0.5*do
    tau = (BETA_R*(od_r*dl)[..., None] + BETA_ME*(od_m*dl)[..., None]
           + BETA_O3*(od_o*dl)[..., None])
    T_view = xp.exp(-tau)                                  # (N,M,T,3)
    T_sun  = transmittance_lookup(rp, mus_p)

    phR = phase_rayleigh(NUc)[..., None, None]
    phM = phase_mie(NUc)[..., None, None]
    src = (BETA_R*dr[..., None]*phR + BETA_MS*dm[..., None]*phM) * T_sun
    src = src + (BETA_R*dr[..., None] + BETA_MS*dm[..., None]) * psi_lookup(rp, mus_p)

    scattering[:, j] = (T_view * src * dl[..., None]).sum(2)

scattering = xp.asarray(scattering, dtype=xp.float32)
print('scattering LUT:', scattering.shape, ' max =', float(asnumpy(scattering.max())))


In [ ]:
# ---------------- 書き出し用 透過率LUT(Webのマッピングで) ----------------
u_x = xp.arange(T_W) / (T_W - 1.0)
u_y = xp.arange(T_H) / (T_H - 1.0)
mu_t = 1.3*u_x - 0.3
h_t  = u_y**2 * (Rt - Rg)
Rm, MUm = xp.meshgrid(Rg + h_t + 1.0, mu_t, indexing='ij')   # (H, W)
transmittance = xp.asarray(transmittance_integrate(Rm, MUm), dtype=xp.float32)
print('transmittance LUT:', transmittance.shape)


In [ ]:
# ---------------- 保存・プレビュー・zip ----------------
import json, os, shutil

OUT = 'lut_out'
os.makedirs(OUT, exist_ok=True)

asnumpy(scattering).astype('<f4').tofile(f'{OUT}/scattering.bin')
asnumpy(transmittance).astype('<f4').tofile(f'{OUT}/transmittance.bin')

manifest = {
    'version': 1,
    'radianceScale': 1.0,
    'transmittance': {'file': 'transmittance.bin', 'width': T_W, 'height': T_H},
    'scattering': {'file': 'scattering.bin', 'muSize': S_MU, 'muSSize': S_MUS, 'nuSize': S_NU},
}
with open(f'{OUT}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

# ---- プレビュー: 太陽高度ごとの「地平線→天頂」カラーストリップ(ACES近似+ガンマ)
import matplotlib.pyplot as plt

def aces(x):
    return np.clip(x*(2.51*x + 0.03) / (x*(2.43*x + 0.59) + 0.14), 0, 1)

S = asnumpy(scattering)
sun_elevs = [60, 30, 10, 2, -2, -6]
fig, axes = plt.subplots(len(sun_elevs), 1, figsize=(8, 6))
for ax, el in zip(axes, sun_elevs):
    mus = np.sin(np.radians(el))
    uj = (1 - np.exp(-3*mus - 0.6)) / (1 - np.exp(-3.6))
    j = int(np.clip(round(uj * (S_MUS - 1)), 0, S_MUS - 1))
    strip = S[S_NU // 2, j][None, :, :]                       # nu≈0 のスライス
    # アプリと同じ露出則 (web/src/app.ts EXPOSURE_CURVE, 太陽強度=20)
    curve = [(6, 0.041), (3, 0.2), (0, 0.51), (-2, 0.9), (-4, 1.41),
             (-6, 2.19), (-8, 2.82), (-10, 3.24), (-12, 3.41), (-14, 3.49), (-18, 3.52)]
    expo = 10 ** np.interp(-el, [-e for e, _ in curve], [l for _, l in curve])
    ax.imshow(aces(strip * 20.0 * expo) ** (1/2.2), aspect='auto',
              extent=[0, 90, 0, 1])
    ax.set_yticks([]); ax.set_ylabel(f'{el:+d}°', rotation=0, labelpad=18)
axes[-1].set_xlabel('view elevation [deg]')
fig.suptitle('sky colour vs sun elevation (multiple scattering)')
fig.savefig(f'{OUT}/preview_strips.png', dpi=110, bbox_inches='tight')
plt.show()

shutil.make_archive('soramado_lut', 'zip', OUT)
print('written:', os.listdir(OUT))

try:
    from google.colab import files
    files.download('soramado_lut.zip')
except Exception:
    print('(not in Colab: soramado_lut.zip is in the working directory)')


## デプロイ / Deploy

1. `soramado_lut.zip` を展開し、`manifest.json` / `transmittance.bin` / `scattering.bin` を
   リポジトリの **`web/public/lut/`** にコミットして再デプロイします。
2. アプリの左上ステータスに「多重散乱LUT」と表示されれば有効化されています
   (無い場合は「単一散乱RT」のままです)。

## 検証のヒント / Sanity checks
- `preview_strips.png` で、太陽高度 +60°→-6° の変化が 青空 → 赤 → マゼンタ → 藍 と物理的に正しく遷移すること
- 多重散乱により、単一散乱と比べて **薄明と地平線付近が明るく自然**(特に日没後)になること

## 将来拡張 / Future work
- 学習ベースのニューラル空モデル(本ノートブックの出力を教師データに学習 → ONNX/テクスチャ化)は、
  Webアプリの `SkySource` インターフェース(`web/src/atmosphere/lut.ts`)から同じ要領で差し替えられます。
